# Benefit-area mapping from pre- and post-project HEC-RAS plans

This example builds and computes a public HEC-RAS 7.0 drainage-improvement scenario, compares hydraulic performance, and creates a categorical benefit-area raster plus polygon output.

The compute stages are part of the workflow. Re-running the notebook skips project extraction and any plan whose result HDF contains HEC-RAS's completed-process marker. Delete `working/612_benefit_area_analysis` to force a clean extraction and full hydraulic re-run.


## Workflow and configuration

The baseline is the Davis storm-system plan. The proposed scenario upsizes four connected high-flow trunk conduits and represents a second identical pump in parallel by doubling the pump-group HQ-curve flow ordinates.

Benefit mapping uses Depth rasters by default. WSE generation is optional because it adds mapping time. Pixel-region filtering and polygon edge simplification are independently optional.


In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display

from ras_commander import (
    BenefitAreaConfig,
    GeomPipeNetwork,
    HdfPipe,
    HdfPump,
    HdfResultsPlan,
    RasBenefits,
    RasCmdr,
    RasExamples,
    RasGeo,
    RasPlan,
    RasProcess,
    init_ras_project,
    ras,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent

WORK_ROOT = REPO_ROOT / "working" / "612_benefit_area_analysis"
PROJECT_SUFFIX = "612_benefit_area"
PROJECT_DIR = WORK_ROOT / f"Davis_{PROJECT_SUFFIX}"
PROJECT_FILE = PROJECT_DIR / "DavisStormSystem.prj"
OUTPUT_DIR = PROJECT_DIR / "BenefitArea" / "p02-to-p01-pipe-pump-upgrade"
RAS_VERSION = "7.0"
BASELINE_PLAN = "02"
PROPOSED_PLAN = "01"
BASELINE_GEOMETRY = "02"
PROPOSED_GEOMETRY = "01"
NUM_CORES = 4

INCLUDE_WSE = False               # True also maps pre/post WSE rasters.
MINIMUM_REGION_PIXELS = 16        # Set to None to disable pixel-region filtering.
SIMPLIFY_POLYGON_EDGES = True     # False preserves exact raster-cell polygon edges.

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Working folder: {WORK_ROOT}")


Working folder: C:\Users\bill\.config\superpowers\worktrees\ras-commander\codex-benefit-area\working\612_benefit_area_analysis


## 1. Extract and initialize the project

`RasExamples.extract_project()` replaces an existing extraction, so the notebook calls it only when the expected project file is absent. This is what makes ordinary re-runs non-destructive.


In [2]:
if PROJECT_FILE.exists():
    project_path = PROJECT_DIR
    print("Extraction skipped: project already exists.")
else:
    project_path = RasExamples.extract_project(
        "Davis",
        output_path=WORK_ROOT,
        suffix=PROJECT_SUFFIX,
    )
    if project_path.resolve() != PROJECT_DIR.resolve():
        raise RuntimeError(f"Unexpected extraction path: {project_path}")
    print(f"Project extracted: {project_path}")

init_ras_project(project_path, RAS_VERSION)
summary_columns = [
    name for name in ("plan_number", "Plan Title", "geometry_number")
    if name in ras.plan_df.columns
]
display(ras.plan_df[summary_columns])


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - ras-commander v0.99.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://rascommander.info | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - Project initialized: DavisStormSystem


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - Using HEC-RAS version 7.0


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/
Repo: https://github.com/gpt-cmdr/ras-commander
LLM agents: https://rascommander.info/llms.txt
═══════════════════════════════════════════════════════════════════════

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow associations
  ras.geom_df        Geometry files and HDF preprocessor paths
  ras.flow_df        Steady flow files
  ras.unsteady_df    Unsteady flow files and configurations
  ras.boundaries_df  Boundary conditions (type, name, location)
  ras.results_df     Lightweight HDF results summaries
  ras.rasmap_df      RASMapper layers, terrain, land cover paths

KEY APIS (static classes — call directly, never instantiate):
  Execution:    RasCmdr.compute_plan() / compute_parallel() / compute_test_mode(

Extraction skipped: project already exists.


,plan_number,Plan Title,geometry_number
0,02,Full System ROM with Pump,02
1,01,Pipe Pump Upgrade,01


## 2. Reusable compute and hydraulic-summary helpers

`skip_existing=True` does more than test for a file: ras-commander verifies the HDF contains the HEC-RAS `Complete Process` marker. A partial or interrupted result therefore proceeds to compute.


In [3]:
def review_compute_messages(plan_number: str, label: str) -> dict:
    messages = HdfResultsPlan.get_compute_messages(plan_number)
    lines = [line.strip() for line in messages.splitlines() if line.strip()]
    if not any("Complete Process" in line for line in lines):
        raise RuntimeError(f"{label} does not contain a completed-process marker")

    fatal = [
        line for line in lines
        if line.upper().startswith(("ERROR:", "FATAL:"))
    ]
    warnings = [line for line in lines if "WARNING" in line.upper()]
    unstable = [
        line for line in lines
        if "UNSTABLE" in line.upper() or "NEGATIVE DEPTH" in line.upper()
    ]
    if fatal:
        raise RuntimeError(f"{label} compute errors:\n" + "\n".join(fatal))

    runtime = HdfResultsPlan.get_runtime_data(plan_number)
    process_hours = (
        float(runtime.iloc[0]["Complete Process (hr)"])
        if runtime is not None and not runtime.empty
        else np.nan
    )
    print(
        f"{label}: complete; runtime={process_hours:.3f} hr; "
        f"warnings={len(warnings)}; instability notices={len(unstable)}"
    )
    for line in (warnings + unstable)[:10]:
        print(f"  {line}")
    return {
        "runtime_hours": process_hours,
        "warning_count": len(warnings),
        "instability_count": len(unstable),
    }


def compute_plan_stage(
    plan_number: str,
    label: str,
    *,
    inputs_changed: bool = False,
) -> dict:
    result_hdf = PROJECT_DIR / f"DavisStormSystem.p{plan_number}.hdf"
    before = result_hdf.stat().st_mtime_ns if result_hdf.exists() else None
    result = RasCmdr.compute_plan(
        plan_number,
        num_cores=NUM_CORES,
        clear_geompre=inputs_changed,
        force_rerun=inputs_changed,
        skip_existing=not inputs_changed,
        verify=True,
    )
    if not result.success:
        raise RuntimeError(f"{label} compute failed")
    after = result_hdf.stat().st_mtime_ns if result_hdf.exists() else None
    executed = before is None or before != after
    print(f"{label} compute stage: {'executed' if executed else 'skipped (completed result exists)'}")
    diagnostics = review_compute_messages(plan_number, label)
    return {"executed": executed, **diagnostics}


def finite_abs_max(values) -> float:
    array = np.asarray(values, dtype=float)
    finite = np.isfinite(array)
    return float(np.max(np.abs(array[finite]))) if finite.any() else np.nan


def pipe_qv_summary(plan_number: str) -> pd.DataFrame:
    conduits = HdfPipe.get_pipe_conduits(plan_number, crs=None).reset_index(drop=True)
    variables = {
        name: HdfPipe.get_pipe_network_timeseries(plan_number, name)
        for name in (
            "Pipes/Pipe Flow US",
            "Pipes/Pipe Flow DS",
            "Pipes/Vel US",
            "Pipes/Vel DS",
        )
    }
    rows = []
    for location, conduit in conduits.iterrows():
        rows.append(
            {
                "conduit": str(conduit["Name"]).strip(),
                "us_node": str(conduit["US Node"]).strip(),
                "ds_node": str(conduit["DS Node"]).strip(),
                "rise_ft": float(conduit["Rise"]),
                "span_ft": float(conduit["Span"]),
                "max_abs_flow_cfs": max(
                    finite_abs_max(variables["Pipes/Pipe Flow US"][:, location]),
                    finite_abs_max(variables["Pipes/Pipe Flow DS"][:, location]),
                ),
                "max_abs_velocity_ft_s": max(
                    finite_abs_max(variables["Pipes/Vel US"][:, location]),
                    finite_abs_max(variables["Pipes/Vel DS"][:, location]),
                ),
            }
        )
    return pd.DataFrame(rows)


def node_depth_summary(plan_number: str) -> pd.DataFrame:
    nodes = HdfPipe.get_pipe_nodes(plan_number, crs=None).reset_index(drop=True)
    depth = HdfPipe.get_pipe_network_timeseries(plan_number, "Nodes/Depth")
    return pd.DataFrame(
        {
            "node": nodes["Name"].astype(str).str.strip(),
            "max_depth_ft": [finite_abs_max(depth[:, i]) for i in range(len(nodes))],
        }
    )


## 3. Compute the baseline and analyze conduit Q/V

The four selected conduits form one connected trunk: 124 → 123 → 129 → 131. They carry some of the largest baseline flows in the network and discharge toward the pump station. The Q/V table is computed before any proposed geometry is created.


In [4]:
baseline_compute = compute_plan_stage(BASELINE_PLAN, "Baseline")
baseline_qv = pipe_qv_summary(BASELINE_PLAN)
baseline_qv.to_csv(WORK_ROOT / "baseline_pipe_qv_summary.csv", index=False)

selected_conduits = ["124", "123", "129", "131"]
selection = baseline_qv[baseline_qv["conduit"].isin(selected_conduits)].copy()
selection["chain_order"] = selection["conduit"].map(
    {name: order for order, name in enumerate(selected_conduits, start=1)}
)
selection = selection.sort_values("chain_order")
display(selection)


2026-07-16 20:11:10 - ras_commander.RasCmdr - INFO - Skipping plan 02: HDF results already exist with 'Complete Process'


Baseline compute stage: skipped (completed result exists)
Baseline: complete; runtime=0.061 hr; warnings=0; instability notices=0


,conduit,us_node,ds_node,rise_ft,span_ft,max_abs_flow_cfs,max_abs_velocity_ft_s,chain_order
9,124,N12-DMH010,N12-DMH009,5.0,5.0,37.990845,2.987258,1
10,123,N12-DMH009,N12-DMH027,5.0,5.0,37.988094,3.168750,2
5,129,N12-DMH027,N13-DMH022,6.0,6.0,53.484386,3.513244,3
3,131,N13-DMH022,O13-SDS03,6.0,6.0,53.747559,3.979787,4


## 4. Build the proposed pipe-and-pump scenario

The conduit setters use absolute dimensions, so calling them again is idempotent. The pump setter likewise writes an absolute HQ curve to both the geometry text file and HDF. If any hydraulic input actually changes, the proposed plan is recomputed; otherwise its completed result is reused.


In [5]:
project_stem = ras.project_name
proposed_geom_file = PROJECT_DIR / f"{project_stem}.g{PROPOSED_GEOMETRY}"
proposed_plan_file = PROJECT_DIR / f"{project_stem}.p{PROPOSED_PLAN}"
scenario_created = False

if not proposed_geom_file.exists():
    new_geom = RasGeo.clone_geom(
        BASELINE_GEOMETRY,
        new_title="Benefit Pipe Pump Upgrade",
    )
    if new_geom != PROPOSED_GEOMETRY:
        raise RuntimeError(f"Expected geometry {PROPOSED_GEOMETRY}; created {new_geom}")
    scenario_created = True
else:
    print("Proposed geometry already exists; clone skipped.")

conduit_targets = {
    "124": (7.5, 7.5),
    "123": (7.5, 7.5),
    "129": (9.0, 9.0),
    "131": (9.0, 9.0),
}
conduit_changes = GeomPipeNetwork.set_conduit_dimensions(
    proposed_geom_file,
    conduit_targets,
)

baseline_pump_groups = HdfPump.get_pump_groups(BASELINE_PLAN)
pump_group_name = "Pump Station #1"
baseline_curve = np.asarray(
    baseline_pump_groups.loc[
        baseline_pump_groups["Name"] == pump_group_name,
        "efficiency_curve",
    ].iloc[0],
    dtype=float,
)
parallel_pump_curve = baseline_curve.copy()
parallel_pump_curve[:, 1] *= 2.0
pump_changes = GeomPipeNetwork.set_pump_group_hq_curve(
    proposed_geom_file,
    pump_group_name,
    parallel_pump_curve,
)

if not proposed_plan_file.exists():
    new_plan = RasPlan.clone_plan(
        BASELINE_PLAN,
        new_shortid="PipePumpUpgrade",
        new_title="Pipe Pump Upgrade",
        geometry=PROPOSED_GEOMETRY,
        num_cores=NUM_CORES,
    )
    if new_plan != PROPOSED_PLAN:
        raise RuntimeError(f"Expected plan {PROPOSED_PLAN}; created {new_plan}")
    scenario_created = True
else:
    print("Proposed plan already exists; clone skipped.")

init_ras_project(PROJECT_DIR, RAS_VERSION)
scenario_changed = bool(
    scenario_created
    or conduit_changes["changed"].any()
    or pump_changes["changed"].any()
)
print(f"Hydraulic scenario inputs changed: {scenario_changed}")
display(conduit_changes)
display(pump_changes)


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - ras-commander v0.99.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://rascommander.info | GitHub: https://github.com/gpt-cmdr/ras-commander


Proposed geometry already exists; clone skipped.
Proposed plan already exists; clone skipped.


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - Project initialized: DavisStormSystem


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - Using HEC-RAS version 7.0


2026-07-16 20:11:10 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/
Repo: https://github.com/gpt-cmdr/ras-commander
LLM agents: https://rascommander.info/llms.txt
═══════════════════════════════════════════════════════════════════════

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow associations
  ras.geom_df        Geometry files and HDF preprocessor paths
  ras.flow_df        Steady flow files
  ras.unsteady_df    Unsteady flow files and configurations
  ras.boundaries_df  Boundary conditions (type, name, location)
  ras.results_df     Lightweight HDF results summaries
  ras.rasmap_df      RASMapper layers, terrain, land cover paths

KEY APIS (static classes — call directly, never instantiate):
  Execution:    RasCmdr.compute_plan() / compute_parallel() / compute_test_mode(

Hydraulic scenario inputs changed: False


,Name,old_rise,old_span,new_rise,new_span,changed
0,124,7.5,7.5,7.5,7.5,False
1,123,7.5,7.5,7.5,7.5,False
2,129,9.0,9.0,9.0,9.0,False
3,131,9.0,9.0,9.0,9.0,False


,group_name,head,old_flow,new_flow,changed
0,Pump Station #1,2.0,140.0,140.0,False
1,Pump Station #1,4.0,120.0,120.0,False
2,Pump Station #1,6.0,110.0,110.0,False
3,Pump Station #1,8.0,100.0,100.0,False
4,Pump Station #1,14.0,80.0,80.0,False
5,Pump Station #1,16.0,50.0,50.0,False


## 5. Compute the proposed plan and compare performance


In [6]:
proposed_compute = compute_plan_stage(
    PROPOSED_PLAN,
    "Proposed",
    inputs_changed=scenario_changed,
)

proposed_qv = pipe_qv_summary(PROPOSED_PLAN)
qv_comparison = (
    baseline_qv.merge(
        proposed_qv,
        on=["conduit", "us_node", "ds_node"],
        suffixes=("_baseline", "_proposed"),
    )
    .query("conduit in @selected_conduits")
    .copy()
)
qv_comparison["delta_max_flow_cfs"] = (
    qv_comparison["max_abs_flow_cfs_proposed"]
    - qv_comparison["max_abs_flow_cfs_baseline"]
)
qv_comparison["delta_max_velocity_ft_s"] = (
    qv_comparison["max_abs_velocity_ft_s_proposed"]
    - qv_comparison["max_abs_velocity_ft_s_baseline"]
)
qv_comparison.to_csv(WORK_ROOT / "proposed_pipe_qv_comparison.csv", index=False)
display(qv_comparison.sort_values("conduit"))

baseline_nodes = node_depth_summary(BASELINE_PLAN)
proposed_nodes = node_depth_summary(PROPOSED_PLAN)
node_comparison = baseline_nodes.merge(
    proposed_nodes,
    on="node",
    suffixes=("_baseline", "_proposed"),
)
node_comparison["depth_reduction_ft"] = (
    node_comparison["max_depth_ft_baseline"]
    - node_comparison["max_depth_ft_proposed"]
)
node_comparison.to_csv(WORK_ROOT / "proposed_node_depth_comparison.csv", index=False)
display(node_comparison.nlargest(10, "depth_reduction_ft"))

pump_rows = []
for plan, condition in ((BASELINE_PLAN, "baseline"), (PROPOSED_PLAN, "proposed")):
    pump_ts = HdfPump.get_pump_station_timeseries(plan, pump_group_name)
    pump_rows.append(
        {
            "condition": condition,
            "peak_flow_cfs": float(pump_ts.sel(variable="Flow").max()),
            "max_headwater_ft": float(pump_ts.sel(variable="Stage HW").max()),
            "max_pumps_on": float(pump_ts.sel(variable="Pumps on").max()),
        }
    )
pump_comparison = pd.DataFrame(pump_rows)
pump_comparison.to_csv(WORK_ROOT / "pump_performance_comparison.csv", index=False)
display(pump_comparison)


2026-07-16 20:11:10 - ras_commander.RasCmdr - INFO - Skipping plan 01: HDF results already exist with 'Complete Process'


Proposed compute stage: skipped (completed result exists)
Proposed: complete; runtime=0.077 hr; warnings=0; instability notices=0


,conduit,us_node,ds_node,rise_ft_baseline,span_ft_baseline,max_abs_flow_cfs_baseline,max_abs_velocity_ft_s_baseline,rise_ft_proposed,span_ft_proposed,max_abs_flow_cfs_proposed,max_abs_velocity_ft_s_proposed,delta_max_flow_cfs,delta_max_velocity_ft_s
10,123,N12-DMH009,N12-DMH027,5.0,5.0,37.988094,3.168750,7.5,7.5,37.141800,3.380770,-0.846294,0.212020
9,124,N12-DMH010,N12-DMH009,5.0,5.0,37.990845,2.987258,7.5,7.5,37.142414,3.186229,-0.848431,0.198971
5,129,N12-DMH027,N13-DMH022,6.0,6.0,53.484386,3.513244,9.0,9.0,60.136116,3.812771,6.651730,0.299528
3,131,N13-DMH022,O13-SDS03,6.0,6.0,53.747559,3.979787,9.0,9.0,62.099358,5.426054,8.351799,1.446267


,node,max_depth_ft_baseline,max_depth_ft_proposed,depth_reduction_ft
4,O15-DMH016,15.302084,13.653049,1.649035
36,P11-DMH024,14.365795,12.798551,1.567245
1,P11-DMH004,14.121315,12.615838,1.505477
5,O15-DMH019,13.719659,12.222616,1.497043
38,P11-DMH026,6.687213,5.296103,1.391110
48,P12-DMH027,4.267089,2.895855,1.371234
0,O14-di027,12.941008,11.577670,1.363338
22,O12-DMH029,6.685491,5.323719,1.361772
39,P11-DMH028,4.656155,3.302805,1.353350
9,O14-DMH030,10.527516,9.199987,1.327529


,condition,peak_flow_cfs,max_headwater_ft,max_pumps_on
0,baseline,67.767029,39.902061,1.0
1,proposed,116.624374,38.253101,1.0


## 6. Generate the BenefitArea raster and polygon

Benefit analysis requires one readable, georeferenced, one-band terrain GeoTIFF shared by both plans. If a consolidated terrain must be created or registered, use `RasTerrain.vrt_to_tiff()`, `RasTerrain.create_terrain_from_rasters()` or `RasTerrain.create_terrain_hdf()`, `RasMap.add_terrain_layer()`, and `RasMap.associate_geometry_layers()` before mapping.

This example maps Depth only. Set `INCLUDE_WSE=True` when pre/post WSE rasters are also needed. Set `MINIMUM_REGION_PIXELS=None` to disable raster-region filtering. Set `SIMPLIFY_POLYGON_EDGES=False` for exact raster-cell polygon edges.


In [7]:
terrain_hdf = PROJECT_DIR / "Terrain" / "Terrain (1).hdf"
terrain_tif = RasBenefits.get_registered_terrain_source(terrain_hdf)
with rasterio.open(terrain_tif) as terrain_src:
    if terrain_src.count != 1:
        raise ValueError("Benefit analysis requires a one-band terrain GeoTIFF")
    terrain_resolution = max(abs(value) for value in terrain_src.res)
    print(
        f"Terrain: {terrain_tif.name}; CRS={terrain_src.crs}; "
        f"resolution={terrain_src.res}"
    )

polygon_tolerance = terrain_resolution if SIMPLIFY_POLYGON_EDGES else None
benefit_config = BenefitAreaConfig(
    pre_plan_number=BASELINE_PLAN,
    terrain_tif=terrain_tif,
    terrain_name="Terrain (1)",
    include_wse=INCLUDE_WSE,
    minimum_region_pixels=MINIMUM_REGION_PIXELS,
    polygon_output=True,
    polygon_simplify_tolerance=polygon_tolerance,
)

benefit_outputs = RasProcess.store_maps(
    plan_number=PROPOSED_PLAN,
    output_path=OUTPUT_DIR,
    profile="Max",
    benefit_area=benefit_config,
    ras_version=RAS_VERSION,
    timeout=1800,
)
for key, paths in benefit_outputs.items():
    print(f"{key}: {[str(path) for path in paths]}")


Terrain: Terrain (1).Davis_terrain2.tif; CRS=EPSG:2871; resolution=(28.84524488389443, 28.84524488389443)


2026-07-16 20:11:13 - rasterio._env - WARNING - CPLE_AppDefined in Depth (Max).Terrain (1).Davis_terrain2.tif: TIFFFetchNormalTag:ASCII value for tag "GDALNoDataValue" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


C:\Users\bill\.config\superpowers\worktrees\ras-commander\codex-benefit-area\.venv\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
2026-07-16 20:11:13 - rasterio._env - WARNING - CPLE_AppDefined in TIFFFetchNormalTag:ASCII value for tag "GDALNoDataValue" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


2026-07-16 20:11:13 - ras_commander.RasProcess - WARNING - Retrying BenefitArea source maps for plan p01 (attempt 2 of 3): Depth TIFF is all NoData while the plan HDF contains wet cells


2026-07-16 20:11:14 - rasterio._env - WARNING - CPLE_AppDefined in Depth (Max).Terrain (1).Davis_terrain2.tif: TIFFFetchNormalTag:ASCII value for tag "GDALNoDataValue" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


...

2026-07-16 20:11:16 - rasterio._env - WARNING - CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


2026-07-16 20:11:16 - pyogrio._io - INFO - Created 3 records


2026-07-16 20:11:16 - ras_commander.RasBenefits - INFO - BenefitArea complete: classes=3; cells=17874; polygon=True


2026-07-16 20:11:16 - ras_commander.RasProcess - INFO - BenefitArea stored-map workflow complete: pre=p02; post=p01; wse=False; filter=16; polygon=True


depth: ['C:\\Users\\bill\\.config\\superpowers\\worktrees\\ras-commander\\codex-benefit-area\\working\\612_benefit_area_analysis\\Davis_612_benefit_area\\BenefitArea\\p02-to-p01-pipe-pump-upgrade\\p01\\Depth (Max).Terrain (1).Davis_terrain2.tif']
benefit_source_pre_depth: ['C:\\Users\\bill\\.config\\superpowers\\worktrees\\ras-commander\\codex-benefit-area\\working\\612_benefit_area_analysis\\Davis_612_benefit_area\\BenefitArea\\p02-to-p01-pipe-pump-upgrade\\p02\\Depth (Max).Terrain (1).Davis_terrain2.tif']
benefit_source_post_depth: ['C:\\Users\\bill\\.config\\superpowers\\worktrees\\ras-commander\\codex-benefit-area\\working\\612_benefit_area_analysis\\Davis_612_benefit_area\\BenefitArea\\p02-to-p01-pipe-pump-upgrade\\p01\\Depth (Max).Terrain (1).Davis_terrain2.tif']
benefit_area: ['C:\\Users\\bill\\.config\\superpowers\\worktrees\\ras-commander\\codex-benefit-area\\working\\612_benefit_area_analysis\\Davis_612_benefit_area\\BenefitArea\\p02-to-p01-pipe-pump-upgrade\\Benefit Area (Ma

## 7. Validate the raster classes and benefit polygons


In [8]:
benefit_raster = benefit_outputs["benefit_area"][0]
benefit_polygon = benefit_outputs["benefit_area_polygon"][0]

category_names = {
    1: "No Change",
    2: "Partially Benefited",
    3: "Fully Benefited",
}
with rasterio.open(benefit_raster) as src:
    values = src.read(1)
    pixel_area = abs(src.transform.a * src.transform.e)
    raster_rows = []
    for code, name in category_names.items():
        count = int(np.count_nonzero(values == code))
        raster_rows.append(
            {
                "benefit_code": code,
                "benefit_status": name,
                "cell_count": count,
                "area_acres": count * pixel_area / 43560.0,
            }
        )
    if not set(np.unique(values)).issubset({0, 1, 2, 3}):
        raise AssertionError("Unexpected BenefitArea raster class")

raster_statistics = pd.DataFrame(raster_rows)
polygons = gpd.read_file(benefit_polygon)
if polygons.empty or not polygons.geometry.is_valid.all():
    raise AssertionError("BenefitArea polygon output is empty or invalid")
polygon_statistics = polygons.drop(columns="geometry")

display(raster_statistics)
display(polygon_statistics)
print(
    f"Validated {len(polygons)} polygon features; "
    f"edge simplification tolerance={polygon_tolerance}; "
    f"pixel filter={MINIMUM_REGION_PIXELS}."
)


C:\Users\bill\AppData\Local\Temp\ipykernel_5232\4224551718.py:10: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  values = src.read(1)


,benefit_code,benefit_status,cell_count,area_acres
0,1,No Change,17093,326.496765
1,2,Partially Benefited,658,12.568588
2,3,Fully Benefited,123,2.349447


,benefit_code,benefit_status,condition,cell_count,area_map_units,area_acres
0,1,No Change,Post Project,17093,1.422220e+07,326.496765
1,2,Partially Benefited,Post Project,658,5.474877e+05,12.568588
2,3,Fully Benefited,Post Project,123,1.023419e+05,2.349447


Validated 3 polygon features; edge simplification tolerance=28.84524488389443; pixel filter=16.


## Re-run behavior

Run the notebook again without deleting the working folder. The extraction cell reports a skip, both compute stages reuse completed results, hydraulic summaries are refreshed, and the BenefitArea maps are regenerated. Delete `working/612_benefit_area_analysis` when a complete extraction and hydraulic re-run is required.
